# 02 — Collaborative Filtering Models

**Models:** Global Average Baseline, User-Based CF, Item-Based CF  
**Libraries:** pandas, numpy, scikit-learn  
**Split:** 80% train / 20% test

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DATA_DIR = '../data/processed/'
RESULTS_DIR        = '../results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Imports successful.')

---
## 2. Load & Split Data

In [ ]:
ratings = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'ratings_clean.csv'))
print(f'Total ratings: {len(ratings):,}')
print(ratings.head())

In [ ]:
train_df, test_df = train_test_split(ratings, test_size=0.20, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train size : {len(train_df):,}')
print(f'Test size  : {len(test_df):,}')

In [ ]:
# Helper: compute RMSE and MAE
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{model_name:<20} | RMSE: {rmse:.4f} | MAE: {mae:.4f}')
    return rmse, mae

---
## 3. Model 1 — Global Average Baseline

Predicts the global mean rating from the training set for every user-movie pair. No personalization — just a lower-bound benchmark.

In [ ]:
global_mean = train_df['rating'].mean()
print(f'Global mean rating: {global_mean:.4f}')

preds_global = np.full(len(test_df), global_mean)
rmse_global, mae_global = evaluate(test_df['rating'], preds_global, 'Global Average')

---
## 4. Build the User-Item Matrix

In [ ]:
# Pivot training data into a user x movie matrix
# Missing entries filled with 0 for similarity computation
user_item_matrix = train_df.pivot_table(
    index='userId', columns='movieId', values='rating'
).fillna(0)

print(f'User-item matrix shape: {user_item_matrix.shape}')
print(f'Users: {user_item_matrix.shape[0]} | Movies: {user_item_matrix.shape[1]}')

---
## 5. Model 2 — User-Based Collaborative Filtering

For each user, find the K most similar users using cosine similarity. Predict the target user's rating as a weighted average of those neighbors' ratings.

In [ ]:
# Compute user-user cosine similarity matrix
user_similarity = cosine_similarity(user_item_matrix)
user_sim_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
print(f'User similarity matrix shape: {user_sim_df.shape}')

In [ ]:
def predict_user_based(user_id, movie_id, user_sim_df, user_item_matrix, global_mean, k=20):
    """
    Predict a rating for (user_id, movie_id) using user-based CF.
    Falls back to global mean if the user or movie is unseen.
    """
    if user_id not in user_sim_df.index or movie_id not in user_item_matrix.columns:
        return global_mean

    sim_scores    = user_sim_df[user_id].drop(index=user_id)
    movie_ratings = user_item_matrix[movie_id]
    rated_mask    = movie_ratings > 0
    sim_scores    = sim_scores[rated_mask]

    if sim_scores.empty or sim_scores.sum() == 0:
        return global_mean

    top_k            = sim_scores.nlargest(k)
    weights          = top_k.values
    neighbor_ratings = user_item_matrix.loc[top_k.index, movie_id].values

    return np.dot(weights, neighbor_ratings) / np.sum(np.abs(weights))

In [ ]:
# Tune K using a validation sample
k_values = [10, 20, 40]
val_df   = test_df.sample(min(500, len(test_df)), random_state=42)

best_k_user, best_rmse_user = 20, float('inf')

for k in k_values:
    preds = val_df.apply(
        lambda row: predict_user_based(
            row['userId'], row['movieId'],
            user_sim_df, user_item_matrix, global_mean, k=k
        ), axis=1
    )
    rmse = np.sqrt(mean_squared_error(val_df['rating'], preds))
    print(f'  k={k:>3}  RMSE: {rmse:.4f}')
    if rmse < best_rmse_user:
        best_rmse_user, best_k_user = rmse, k

print(f'\nBest k for User-Based CF: {best_k_user}')

In [ ]:
# Evaluate on full test set
preds_user_cf = test_df.apply(
    lambda row: predict_user_based(
        row['userId'], row['movieId'],
        user_sim_df, user_item_matrix, global_mean, k=best_k_user
    ), axis=1
)

rmse_user_cf, mae_user_cf = evaluate(test_df['rating'], preds_user_cf, 'User-Based CF')

---
## 6. Model 3 — Item-Based Collaborative Filtering

For each movie, find the K most similar movies using cosine similarity. Predict by taking a weighted average of the user's ratings on those similar movies.

In [ ]:
# Compute item-item cosine similarity (transpose so movies are rows)
item_similarity = cosine_similarity(user_item_matrix.T)
item_sim_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)
print(f'Item similarity matrix shape: {item_sim_df.shape}')

In [ ]:
def predict_item_based(user_id, movie_id, item_sim_df, user_item_matrix, global_mean, k=20):
    """
    Predict a rating for (user_id, movie_id) using item-based CF.
    Falls back to global mean if user or movie is unseen.
    """
    if user_id not in user_item_matrix.index or movie_id not in item_sim_df.index:
        return global_mean

    sim_scores   = item_sim_df[movie_id].drop(index=movie_id)
    user_ratings = user_item_matrix.loc[user_id]
    rated_mask   = user_ratings > 0
    sim_scores   = sim_scores[rated_mask]

    if sim_scores.empty or sim_scores.sum() == 0:
        return global_mean

    top_k            = sim_scores.nlargest(k)
    weights          = top_k.values
    neighbor_ratings = user_item_matrix.loc[user_id, top_k.index].values

    return np.dot(weights, neighbor_ratings) / np.sum(np.abs(weights))

In [ ]:
# Tune K
best_k_item, best_rmse_item = 20, float('inf')

for k in k_values:
    preds = val_df.apply(
        lambda row: predict_item_based(
            row['userId'], row['movieId'],
            item_sim_df, user_item_matrix, global_mean, k=k
        ), axis=1
    )
    rmse = np.sqrt(mean_squared_error(val_df['rating'], preds))
    print(f'  k={k:>3}  RMSE: {rmse:.4f}')
    if rmse < best_rmse_item:
        best_rmse_item, best_k_item = rmse, k

print(f'\nBest k for Item-Based CF: {best_k_item}')

In [ ]:
# Evaluate on full test set
preds_item_cf = test_df.apply(
    lambda row: predict_item_based(
        row['userId'], row['movieId'],
        item_sim_df, user_item_matrix, global_mean, k=best_k_item
    ), axis=1
)

rmse_item_cf, mae_item_cf = evaluate(test_df['rating'], preds_item_cf, 'Item-Based CF')

---
## 7. Results Summary

In [ ]:
results_cf = pd.DataFrame({
    'Model': ['Global Average', 'User-Based CF', 'Item-Based CF'],
    'RMSE':  [round(rmse_global, 4), round(rmse_user_cf, 4), round(rmse_item_cf, 4)],
    'MAE':   [round(mae_global, 4),  round(mae_user_cf, 4),  round(mae_item_cf, 4)]
})

print(results_cf.to_string(index=False))

results_cf.to_csv(os.path.join(RESULTS_DIR, 'metrics_cf.csv'), index=False)
print('\nSaved metrics_cf.csv')